# Decaying Random Flow, Walled Box (2D)

The same divergence-free Perlin noise field as `randomFlow_periodic.ipynb`, from
the same seed, in a box with **free-slip walls** instead of periodic sides. The
pair exists for the comparison: periodic decay is the scheme's own dissipation
plus the physical viscosity, and this one adds whatever the boundary
contributes.

Two things are specific to the walled variant:

- **`band` is what makes the walls exist.** `--bounded` cuts the wall region out
  of the *interior* domain, and `band` is the number of particle layers by which
  the simulated box is wider than that interior. At `band=0` -- the periodic
  default -- the wall region encloses no volume and **zero boundary particles
  are sampled**, i.e. `--bounded` silently runs the periodic case. The case
  supplies `band=5` when it is left at 0 (`randomFlow.BOUNDED_BAND`); the
  parameters cell below names it explicitly anyway, because it is not a knob to
  discover the hard way.
- **The density near a wall is the thing to look at.** Boundary particles are
  sampled on their own lattice, so a wall is where an SPH density sum is most
  likely to come out wrong. The cell after the IC build re-evaluates the density
  directly from the kernel sum and plots it, which is a check the live panels
  (which show the *evolved* density) cannot give you at t = 0.

![](outputs/06-randomFlowBounded.gif)

## Every knob, and what it does

The parameters cell below is the whole command line of `randomFlow_bounded.py` written
out: `CaseSpec` fields first, then `randomFlowCase.params` -- the case's own
physics knobs, each of which is also a `--flag`. Anything not named there keeps
the value in `randomFlowCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `256` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `2.0` | side of the box |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `10.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `60` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `bounded` | `True` | free-slip walls cut from the interior domain |
| `band` | `5` | particle layers of wall padding -- **at 0 the walls have zero volume and are not sampled at all** |
| `obstacle` | `False` | no rigid body here; the periodic sibling notebook has one |
| `obstacleShape` | `'circle'` | any key of `SHAPE_PRESETS`: `circle`, `box`, `roundedBox`, `rhombus`, `trapezoid`, `parallelogram`, `equilateralTriangle`, `triangleIsosceles`, `pentagon`, `hexagon`, `octogon`, `hexagram`, `star5`, `vesica`, `cutDisk`, `unevenCapsule`, `moon` |
| `obstacleSize` | `0.25` | characteristic half-size of that body |
| `obstacleAspect` | `1.0` | squashes it in its second direction |
| `obstacleRotation` | `0.0` | degrees counter-clockwise, its orientation |
| `obstacleOffset` | `[0.0, 0.0]` | where its (measured) centre sits; a list, so `--config`/notebook only |
| `octaves` | `3` | how many frequencies of noise are summed |
| `baseFrequency` | `2` | the largest eddy, in periods across the box |
| `lacunarity` | `2` | frequency ratio between successive octaves |
| `persistence` | `0.5` | amplitude ratio between them, so higher octaves are weaker |
| `tileable` | `True` | make the noise periodic, which it has to be in a periodic box |
| `kind` | `'perlin'` | noise family |
| `seed` | `45906734` | the seed; both notebooks of this pair use it, so the two runs start from the same field |
| `bandWidth` | `16.0` | surface-detection bandwidth, in particle spacings |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.00025` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `False`, `0.005` | physical viscosity; without it the decay rate is set by the scheme's own dissipation, which `05-taylor-green-vortex.ipynb` measures |
| `alpha` | `0.01` | artificial-viscosity coefficient; unused while `inviscid=False`, but it is what `nu` converts to |
| `freeSurface` | `False` | surface detection, on for a case with a free surface |
| `markerSize` | `4` | plot only: particle marker size |

**Three things this family does differently from the compressible notebooks**
(they will bite if `../../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `randomFlowCase.initialConditions(ctx, system)`.
   That is where the noise field is sampled onto the particles, and where
   `setupWeaklyCompressibleTimestep` picks the sound speed and `config.dt`
   *together* from `targetDt` -- weakly compressible SPH is free to choose its
   own stiffness, so the timestep is fixed first and `c0` follows from the
   acoustic CFL. Skip it and `config.dt` stays `None`, and the box is
   motionless.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `randomFlowCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.randomFlow import randomFlowCase
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame
from warpSPHCore import (GradientScheme, OperationProperties, SupportScheme,
                         WarpOperation, warpOperation)

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `randomFlow_bounded.py`, made
# explicit and editable here -- the table in the intro cell says what each one
# does. `randomFlowCase.defaults`/`.params` are the same values the CLI script
# starts from.
spec = CaseSpec(caseName=randomFlowCase.name, scheme=randomFlowCase.scheme,
                params=dict(randomFlowCase.params)) \
    .merged(**randomFlowCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=256,
    dim=2,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,

    # --- output --------------------------------------------------------------
    caseName='06-randomFlowBounded',
    plot=True, show=True, plotInterval=60,
    store=False,

    # --- the flow's own knobs -------------------------------------------------
    params=dict(
        # the box: periodic sides, or walls cut from the interior domain
        bounded=True, band=5,
        # an optional rigid body: any shape from SHAPE_PRESETS, placed and turned
        obstacle=False, obstacleShape='circle', obstacleSize=0.25,
        obstacleAspect=1.0, obstacleRotation=0.0, obstacleOffset=[0.0, 0.0],
        # the initial field: divergence-free Perlin noise, same seed in both
        # notebooks of this pair
        octaves=3, lacunarity=2, persistence=0.5, baseFrequency=2,
        tileable=True, kind='perlin', seed=45906734,
        bandWidth=16.0,
        # the fluid
        rho0=1.0, targetDt=0.00025, inviscid=False, nu=0.005, alpha=0.01,
        freeSurface=False,
        markerSize=4,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`randomFlowCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have: the
# divergence-free noise field is sampled onto the particles there, and it is
# where the sound speed and `config.dt` are chosen together from `targetDt`, so
# skipping it leaves `config.dt` unset and the box motionless.
ctx = buildContext(randomFlowCase, spec)
randomFlowCase.configureScheme(ctx)
system = randomFlowCase.buildSystem(ctx)
randomFlowCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

kinds = system.state.kinds
print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles '
      f'({int((kinds == 0).sum())} fluid, {int((kinds != 0).sum())} boundary)')

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# geometry parameter above did something other than what it sounded like -- and
# for this case in particular, it is where "the walls are missing" is visible
# before the run rather than after it.
figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

## Is the density right where the walls are?

The live density panel shows the *evolved* density, which is a scheme output.
This cell instead re-evaluates the plain SPH density sum $\rho_i = \sum_j m_j
W_{ij}$ on the initial state, which is a statement about the **sampling** -- and
a wall is where sampling is most likely to be wrong.

At the shipped `band=5` it comes out flat: `rho0` to within about `1e-4`
everywhere, wall particles included, because the walls are sampled on the same
lattice as the fluid and five layers is more than the four-spacing support
radius needs. That is the answer you want, and it is worth seeing *because* it
is not automatic -- set `band=1` in the parameters cell and re-run this cell,
and the whole box (not just the wall) picks up a percent-level error, since the
domain is still flagged periodic and a wall thinner than the support lets
particles see straight across the wrap.

In [ ]:
density = warpOperation(
    runningState.state,
    OperationProperties(
        kernel=ctx.config.kernel,
        operation=WarpOperation.Density,
        # Gather, matching what the density-estimate literature (Cullen &
        # Dehnen 2010, appendix E.1) does for a diagnostic sum like this one.
        supportMode=SupportScheme.Gather,
    ),
    domain=ctx.config.domain,
    adjacency=runningState.adjacency,
)

positions = runningState.state.positions.detach().cpu().numpy()
# float64 for the histogram: at band=5 the whole spread is ~1e-6, which is
# below float32's spacing at 1.0, so 80 float32 bin edges are not distinct.
values = density.detach().cpu().numpy().astype(np.float64)
fluid = (runningState.state.kinds == 0).detach().cpu().numpy()
print(f'rho: fluid [{values[fluid].min():.6f}, {values[fluid].max():.6f}], '
      f'boundary [{values[~fluid].min():.6f}, {values[~fluid].max():.6f}]')

figure, axis = plt.subplots(1, 2, figsize=(12, 4.5))
scatter = axis[0].scatter(positions[:, 0], positions[:, 1], c=values, s=1, cmap='viridis')
figure.colorbar(scatter, ax=axis[0])
axis[0].set_aspect('equal')
axis[0].set_title(r'$\sum_j m_j W_{ij}$ on the initial state')

bins = np.linspace(values.min(), values.max(), 80) if np.ptp(values) > 0 else 1
axis[1].hist(values[fluid], bins=bins, label='fluid')
axis[1].hist(values[~fluid], bins=bins, alpha=0.6, label='boundary')
axis[1].axvline(spec.param('rho0'), color='black', ls=':', lw=0.8, label=r'$\rho_0$')
axis[1].set_xlabel(r'$\rho$'); axis[1].set_yscale('log'); axis[1].legend()
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not randomFlowCase.setupPlot --
# see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = randomFlowCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=randomFlowCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = [dict(randomFlowCase.diagnostics(ctx, runningState), step=-1, t=0.0)]
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = randomFlowCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=randomFlowCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## What decayed, and did the fluid stay a fluid?

There is no closed-form decay law here -- that is `05-taylor-green-vortex.ipynb`,
which is the one case in this family whose dissipation can be checked against an
analytic answer. What this pair gives instead is a *comparison*: the same field,
the same seed, the same viscosity model, run with and without walls. Plot the
two `kineticEnergy` series against each other and the difference is what the
boundary did.

The right-hand panel is the standing health check for every weakly compressible
case: the density bounds against the +-1% band the scheme's whole premise rests
on. Excursions at the walls (or around the obstacle) show up here first.

In [ ]:
figure, axis = plt.subplots(1, 2, figsize=(11, 4))
t = np.array([row['t'] for row in trajectory])
energy = np.array([row['kineticEnergy'] for row in trajectory])

axis[0].plot(t, energy / energy[0])
axis[0].set_xlabel('t'); axis[0].set_ylabel(r'$E_k / E_k(0)$'); axis[0].set_yscale('log')
axis[0].set_title(f"decay, nu = {spec.param('nu')}")

axis[1].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[1].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()

# Saved into this run's export directory (not next to the notebook -- `export/`
# is gitignored) so the sibling variant can be overlaid on it:
# `np.load('<the path printed here>')` in the other notebook.
energyPath = os.path.join(ctx.exportPath, 'kineticEnergy.npz')
np.savez(energyPath, t=t, kineticEnergy=energy)
print(f'wrote {energyPath}')